In [ ]:
import numpy as np
import pandas as pd
import yaml
import re
import json

from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
df_firms = pd.read_csv(dataset_config['path_processed'] + 'WOS/03_WOS_unique_orgs_firms_US_filtered.csv')
df_firms = df_firms[df_firms.hq_match == 1][['organization']]
df_firms

In [ ]:
from pathlib import Path
from tqdm import tqdm
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")
client = OpenAI()

In [ ]:
N_PER_FILE = 10_000  # shard size
MODEL = "gpt-4o"

PROMPT_INSTRUCTIONS = (
    "Return 1 if it is a US company; otherwise return 0. "
    "If uncertain, return 0."
)

# Prepare output directory
OUTPUT_DIR = Path(dataset_config["path_processed"]) / "WOS" / "_CHAT"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------
# Create records from df
# --------------------------------------------------------------------
records = [
    {"organization": str(x)}
    for x in df_firms["organization"].dropna().tolist()
]

In [ ]:
import math

def write_jsonl_batch(records, start, end, out_dir: Path):
    shard = records[start:end]
    shard_id = start // N_PER_FILE
    out_path = out_dir / f"batch_{shard_id:02d}.jsonl"

    if out_path.exists():
        print(f"[skip] {out_path.name} already exists.")
        return out_path

    with out_path.open("w", encoding="utf-8") as f:
        for i, r in enumerate(
            tqdm(shard, desc=f"[write] {out_path.name}", leave=False),
            start=start
        ):
            org = r["organization"]

            req = {
                "custom_id": f"org-{i}",
                "method": "POST",
                "url": "/v1/responses",
                "body": {
                    "model": MODEL,
                    "instructions": PROMPT_INSTRUCTIONS,
                    "input": org,
                },
            }

            f.write(json.dumps(req, ensure_ascii=False) + "\n")

    print(f"[done] Wrote {end - start:,} records → {out_path}")
    return out_path

num_shards = math.ceil(len(records) / N_PER_FILE)
for s in range(num_shards):
    a = s * N_PER_FILE
    b = min((s + 1) * N_PER_FILE, len(records))
    write_jsonl_batch(records, a, b, OUTPUT_DIR)

print(f"\nAll {num_shards} JSONL shard files written → {OUTPUT_DIR.resolve()}")